# Qwen NER SFT Notebook


使用完整训练集 `llm_ner_dataset2/output/train.json` 做 SFT，并使用 `dev.json` 做评估，先让模型学会更稳定地输出目标 JSON 格式。

这里先不做 GRPO，只做监督微调。

In [60]:
from pathlib import Path
import json
import torch

PROJECT_ROOT = Path.cwd()
MODEL_PATH = Path(r"C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B-Instruct")
TRAIN_PATH = PROJECT_ROOT / "llm_ner_dataset2" / "output" / "train.json"
DEV_PATH = PROJECT_ROOT / "llm_ner_dataset2" / "output" / "dev.json"
SFT_RUN_DIR = PROJECT_ROOT / "sft_runs"
SFT_RUN_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("DEV_PATH exists:", DEV_PATH.exists())
print("cuda available:", torch.cuda.is_available())


PROJECT_ROOT: D:\github\Reinforcement-Learning\GRPO
MODEL_PATH exists: True
TRAIN_PATH exists: True
DEV_PATH exists: True
cuda available: True


In [61]:
def load_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

train_data = load_jsonl(TRAIN_PATH)
dev_data = load_jsonl(DEV_PATH)
print("train size:", len(train_data))
print("dev size:", len(dev_data))
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


train size: 18000
dev size: 3496
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [
      "浙商银行"
    ],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  ],
  "text": "浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
}


In [62]:
# 这里不做“全文监督”，而是做 response-only SFT。
# 也就是说：
# 1. system + user 只作为条件输入
# 2. 只有 assistant 的标准 JSON 回复参与 loss
#
# 这样更符合当前任务目标：
# 模型需要学习的是“如何输出正确 JSON”，
# 而不是去背 system / user 这部分 prompt。

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

def build_sft_example(sample):
    # 标准答案就是 assistant 应该输出的 JSON 字符串
    answer_text = json.dumps(sample["answer"], ensure_ascii=False)

    # prompt_text：只保留 system + user，并停在“等待 assistant 回复”的位置
    # 这里 add_generation_prompt=True 很关键，
    # 因为我们希望模型看到的是“到 assistant 开始生成前”的上下文。
    prompt_text = tokenizer.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True,
    )

    # full_text：真正喂给模型训练的完整文本 = prompt + 正确 assistant 回复
    full_text = prompt_text + answer_text

    return {
        "prompt_text": prompt_text,
        "answer_text": answer_text,
        "full_text": full_text,
        "schema": sample["schema"],
        "raw_text": sample["text"],
    }

train_sft_data = [build_sft_example(x) for x in train_data]
dev_sft_data = [build_sft_example(x) for x in dev_data[:200]]
print("train sft size:", len(train_sft_data))
print("dev sft size:", len(dev_sft_data))
print(train_sft_data[0]["prompt_text"][:800])
print("-" * 80)
print(train_sft_data[0]["answer_text"])


train sft size: 18000
dev sft size: 200
<|im_start|>system
你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。<|im_end|>
<|im_start|>user
请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。

schema: ["address", "book", "company", "game", "government", "movie"]
text: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，<|im_end|>
<|im_start|>assistant

--------------------------------------------------------------------------------
{"address": [], "book": [], "company": ["浙商银行"], "game": [], "government": [], "movie": []}


In [63]:
# 这里优先尝试用 datasets.Dataset，便于喂给 Trainer。

from datasets import Dataset

# 这里改成完整训练集训练，dev 集评估。
# 如果你后面觉得太慢，再手动把下面两行改回较小子集。
train_subset_size = len(train_sft_data)
eval_subset_size = len(dev_sft_data)

train_subset = train_sft_data[:train_subset_size]
eval_subset = dev_sft_data[:eval_subset_size]

train_dataset = Dataset.from_list(train_subset)
eval_dataset = Dataset.from_list(eval_subset)

print(train_dataset)
print(eval_dataset)


Dataset({
    features: ['prompt_text', 'answer_text', 'full_text', 'schema', 'raw_text'],
    num_rows: 18000
})
Dataset({
    features: ['prompt_text', 'answer_text', 'full_text', 'schema', 'raw_text'],
    num_rows: 200
})


In [64]:
# response-only SFT。
# 核心思想：
# 1. input_ids 仍然是完整文本：prompt + 正确 assistant 回复
# 2. 但 labels 里，prompt 部分全部设为 -100
# 3. 只有 assistant 回复部分保留真实 token id，参与 loss
#
# 这样训练时，模型不会因为 system / user 部分被反向传播，
# 而是专注学习“该怎么生成目标 JSON 回复”。

max_length = 512

def tokenize_fn(example):
    # prompt_text：只到 assistant 开始生成前
    prompt_text = example["prompt_text"]

    # full_text：prompt + 正确 assistant 回复
    full_text = example["full_text"]

    # 先分别编码 prompt 和 full_text
    # add_special_tokens=False，避免这里额外插入 token，保持两段长度关系清晰
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_length,
    )["input_ids"]

    # 如果 full_ids 因为 max_length 被截断了，
    # prompt 部分长度也必须同步裁剪到 full_ids 的实际长度以内
    prompt_len = min(len(prompt_ids), len(full_ids))

    attention_mask = [1] * len(full_ids)

    # response-only labels：
    # - prompt 部分全部设为 -100，不参与 loss
    # - assistant 回复部分保留真实 token id，参与 loss
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_tokenized = train_dataset.map(tokenize_fn, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(tokenize_fn, remove_columns=eval_dataset.column_names)

print(train_tokenized)
print(eval_tokenized)

# 看一条样本，确认 labels 前面一段确实是 -100
print(train_tokenized[0]["labels"][:80])


Map: 100%|██████████| 200/200 [00:00<00:00, 2657.18 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 18000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [65]:
# 走 LoRA
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1165.96it/s]


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [66]:
from transformers import TrainingArguments, Trainer

# 这里不能再用 DataCollatorForLanguageModeling。
# 原因是：
# 1. 我们已经自己手工构造好了 response-only labels
# 2. 每条样本的 labels 长度不一样
# 3. DataCollatorForLanguageModeling 更适合“由它自己生成 labels”的场景
#
# 所以这里改成自定义 collator：
# - input_ids 按 tokenizer.pad_token_id 补齐
# - attention_mask 按 0 补齐
# - labels 按 -100 补齐
#
# 这样 prompt 部分和 padding 部分都不会参与 loss。
def response_only_collator(features):
    max_len = max(len(x["input_ids"]) for x in features)

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for x in features:
        input_ids = x["input_ids"]
        attention_mask = x["attention_mask"]
        labels = x["labels"]

        pad_len = max_len - len(input_ids)

        batch_input_ids.append(input_ids + [tokenizer.pad_token_id] * pad_len)
        batch_attention_mask.append(attention_mask + [0] * pad_len)
        batch_labels.append(labels + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
        "labels": torch.tensor(batch_labels, dtype=torch.long),
    }

training_args = TrainingArguments(
    output_dir=str(SFT_RUN_DIR / "qwen_ner_sft_train"),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=5,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=100,
    bf16=torch.cuda.is_available(),
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=response_only_collator,
)

print(trainer)


In [67]:
train_result = trainer.train()
print(train_result)


Step,Training Loss,Validation Loss
100,0.144515,0.130357
200,0.127601,0.108061
300,0.131099,0.098079
400,0.097196,0.092122
500,0.111496,0.088861
563,0.111800,0.088322


TrainOutput(global_step=563, training_loss=0.14042054177601002, metrics={'train_runtime': 567.1659, 'train_samples_per_second': 31.737, 'train_steps_per_second': 0.993, 'total_flos': 5535800017259520.0, 'train_loss': 0.14042054177601002, 'epoch': 1.0})


In [68]:
# 保存 LoRA 权重和 tokenizer

save_dir = SFT_RUN_DIR / "qwen_ner_sft_dev_final"
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print("saved to:", save_dir)


saved to: D:\github\Reinforcement-Learning\GRPO\sft_runs\qwen_ner_sft_dev_final


In [70]:
# 冒烟测试一下微调后的模型，看看能不能正确输出 JSON 格式的回复。

example_idx = 2
demo_sample = dev_data[example_idx]

chat_text = tokenizer.apply_chat_template(
    demo_sample["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False,
    )

completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
response = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

print("text:", demo_sample["text"])
print("schema:", demo_sample["schema"])
print("gold:", json.dumps(demo_sample["answer"], ensure_ascii=False))
print("pred:", response)


text: 这也是我们人居环境委员会从住区向城镇发展的一个非常重要的标志。在这个体系里面，
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": [], "book": [], "company": [], "game": [], "government": ["人居环境委员会"], "movie": []}
pred: {"address": [], "book": [], "company": [], "game": [], "government": ["人居环境委员会"], "movie": []}
